# A Computational Assessment of Social Sector Spending Across Indian States (2013-14 to 2025-26)
## CAG-ICSSR Research Initiative: Trend, Cluster, and Anomaly Analysis

This notebook executes the complete end-to-end analytical pipeline on audited CAG State Finance data across 31 States & UTs over 13 fiscal years.

In [1]:
import sys, os
import numpy as np
import pandas as pd

# Add src to path
sys.path.append(os.path.abspath(".."))
from src.trend_analysis.trend_engine import TrendAnalysisEngine
from src.clustering.cluster_engine import ClusterAnalysisEngine
from src.anomaly_detection.anomaly_engine import AnomalyDetectionEngine
from src.econometrics.panel_regressor import PanelRegressionEngine

panel_path = os.path.abspath("../data/processed/panel_2013_2026.csv")
print(f"Panel dataset loaded from: {panel_path}")

### 1. Trend and Convergence Dynamics (RQ1 & RQ2)

In [2]:
trend_engine = TrendAnalysisEngine(panel_path)
trends = trend_engine.analyze_state_trends("sse_pc")
conv = trend_engine.analyze_convergence("sse_pc")
comp_shifts = trend_engine.analyze_composition_shift()

print("--- Convergence Results ---")
print("Beta Convergence (Full Period):", conv["beta_convergence_full"])
print("Composition Dissimilarity (COVID Shock):", comp_shifts["dissimilarity_index_covid_shock"])

### 2. Unsupervised State Clustering (RQ3)

In [3]:
cluster_engine = ClusterAnalysisEngine(panel_path)
eval_metrics = cluster_engine.evaluate_cluster_numbers(k_min=2, k_max=6)
clusters = cluster_engine.run_clustering(n_clusters=4, algorithm="kmeans")

for c_id, profile in clusters["clusters"].items():
    print(f"
Cluster {c_id} ({profile["size"]} states):")
    print("States:", ", ".join(profile["states"][:6]))
    print("Mean Real SSE PC (Rs):", profile["mean_metrics"]["sse_pc_mean"])
    print("Own Tax Effort (OTR/GSDP %):", profile["mean_metrics"]["otr_gsdp_mean"])

### 3. Anomaly Detection & Triangulation Register (RQ4 & RQ5)

In [4]:
anomaly_engine = AnomalyDetectionEngine(panel_path)
register = anomaly_engine.triangulate_anomalies()
print(f"Total flagged anomalies triangulated: {len(register)}")

# Display sample register items
for item in register[:5]:
    print(f"{item["state_name"]} ({item["fiscal_year"]}) -> {item["triangulation_category"]}")
    print(f"  Explanation: {item["context_explanation"]}")
    print(f"  Audit Recommendation: {item["audit_recommendation"]}
")

### 4. Econometric Panel Regressions (Fixed Effects vs Random Effects & Hausman Test)

In [5]:
reg_engine = PanelRegressionEngine(panel_path)
model = reg_engine.estimate_model(dep_var="ln_sse_pc")

print(f"Within R-squared: {model["within_r2"]}")
print(f"Hausman Test Chi2: {model["hausman_test"]["chi2_stat"]} (p = {model["hausman_test"]["p_value"]})")
print(f"Preferred Model: {model["hausman_test"]["preferred_model"]}
")
print("Fixed Effects Estimation Table:")
for res in model["fixed_effects_results"]:
    print(f"{res["variable"]:15}: Coef = {res["coef"]:8.4f}, SE = {res["std_err"]:6.4f}, p = {res["p_value"]:7.5f} {res["significance"]}")